In [1]:
# load the RAG file and the gold file and make sure one gold is always in RAG for training for the answerables

ragfile = "/dccstor/srosent1/human_ai_eval/longNQEval/retrieval/model_runner_format/3-ctx/train/intfloate5-base-v2_results.jsonl"
goldfile = "/dccstor/srosent2/generative/appen/final/longNQ_hf/train/longNQ_train_answerable.jsonl"

In [2]:
import json

def read_jsonl(filename: str, encoding="utf-8"):
    with open(filename, mode="r", encoding=encoding) as fp:
        content = [json.loads(line.rstrip("\n").strip()) for line in fp]

    return content

def dump_jsonl(filename: str, data, encoding="utf-8"):
    with open(filename, mode="w", encoding=encoding) as fp:
        for line in data:
            json.dump(line, fp)
            fp.write("\n")

In [3]:
from rouge_score import rouge_scorer

rouge = rouge_scorer.RougeScorer(rouge_types=['rougeLsum'], split_summaries=True)

In [4]:
# since we will only be using top 3, make sure to put randomly in top 3.

rag_data = read_jsonl(ragfile)
gold_data = read_jsonl(goldfile)
gold_data_by_id = {}

for example in gold_data:
    gold_data_by_id[example['id']] = example

In [5]:
# model runner

import random

checked = 0
adjusted = 0
adjusted_rouge = 0

for example in rag_data:
    if example['task_id'] not in gold_data_by_id:
        continue
    gold_example = gold_data_by_id[example['task_id']]
    random_num = random.randint(0,2)
    checked += 1
    in_top_n = False
    for passage in example['contexts'][:3]:
        if gold_example['passages'][0]['text'].strip() == passage['text'].strip():
            in_top_n = True
            break
        elif rouge.score(gold_example['passages'][0]['text'].strip(), passage['text'].strip())['rougeLsum'][2] >= .9:
            in_top_n = True
            adjusted_rouge += 1
            break

    if not in_top_n:
        example['contexts'][random_num] = {'document_id': 'gold_doc_id', 'title': gold_example['passages'][0]['title'], 'text': gold_example['passages'][0]['text']}
        adjusted += 1

print(f"Adjusted: {adjusted}/{checked} ({adjusted_rouge})")

Adjusted: 719/1954 (38)


In [6]:
dump_jsonl("/dccstor/srosent1/human_ai_eval/longNQEval/retrieval/model_runner_format/3-ctx/train/gold_intfloate5-base-v2_results.jsonl", rag_data)

In [22]:
import random

checked = 0
adjusted = 0
adjusted_rouge = 0

for example in rag_data:
    if example['id'] not in gold_data_by_id:
        continue
    gold_example = gold_data_by_id[example['id']]
    random_num = random.randint(0,2)
    checked += 1
    in_top_n = False
    for passage in example['contexts'][:3]:
        if gold_example['passages'][0]['text'].strip() == passage['text'].strip():
            in_top_n = True
            break
        elif rouge.score(gold_example['passages'][0]['text'].strip(), passage['text'].strip())['rougeLsum'][2] >= .9:
            in_top_n = True
            adjusted_rouge += 1
            break

    if not in_top_n:
        example['passages'][random_num] = gold_example['passages'][0]
        adjusted += 1

print(f"Adjusted: {adjusted}/{checked} ({adjusted_rouge})")

1
1
2
0
1
1
2
0
2
1
0
0
1
2
2
0
0
1
2
0
0
2
2
1
2
1
0
1
1
2
1
0
0
0
1
2
2
1
0
0
2
0
1
0
0
2
0
2
0
1
2
1
0
2
1
2
1
0
2
2
0
1
1
2
1
2
1
1
0
2
2
0
0
2
2
2
0
1
0
2
0
2
1
2
2
2
2
0
2
0
1
1
2
0
0
1
0
1
2
2
0
1
2
0
0
0
0
1
2
0
2
1
2
2
1
1
1
0
0
0
1
1
2
2
1
0
1
2
0
0
2
2
0
0
0
0
0
1
0
1
0
1
0
1
1
0
2
0
2
1
1
0
2
0
2
0
2
1
1
2
1
0
1
1
2
1
1
0
1
1
0
1
0
2
2
0
1
2
1
2
2
2
0
0
0
1
0
1
1
0
1
2
1
2
0
2
1
2
2
2
2
2
0
2
1
2
0
1
0
1
1
2
2
2
2
0
1
1
0
0
0
0
2
1
0
2
2
1
1
0
0
0
1
0
2
2
2
2
2
2
1
1
0
1
0
1
0
2
1
0
1
1
0
2
0
2
2
0
2
1
2
1
0
2
1
1
2
1
0
0
1
2
2
1
0
2
0
1
1
0
0
1
0
2
2
1
0
2
2
0
1
1
1
1
1
1
2
2
1
0
0
2
2
2
0
1
2
1
2
2
2
2
1
1
0
2
0
1
1
0
0
2
2
0
2
2
2
2
1
0
1
1
1
2
1
1
0
2
0
2
1
1
2
0
0
0
1
0
1
0
2
0
0
1
0
2
1
2
2
1
1
1
2
2
0
2
1
2
1
1
1
1
1
2
1
2
2
0
2
1
1
2
2
0
1
2
0
1
2
2
0
1
1
0
1
0
2
0
1
2
2
0
2
2
0
2
0
1
0
0
0
2
1
0
0
0
2
2
0
1
2
1
0
0
2
2
1
0
0


KeyboardInterrupt: 

In [21]:
dump_jsonl("/dccstor/srosent1/human_ai_eval/longNQEval/retrieval/eli5format/5-ctx/train/gold_intfloate5-base-v2_results.jsonl", rag_data)